# S3 — profile: time bins for the slider + $/acre coverage in land ads
Task `tasks/T-004-time-bins-ad-prices.md` (brief B-015). Read-only: SQL + regex only, no ML/LLM. Nothing is written to `data/`.

**Sources**
- Moves, households: `data/processed/residences.parquet`, `households.parquet` through the `moves` view in `rx-pipeline/schema.sql` (A08 outputs, 2026-11-22).
- Land-sale blocks: `RevX_OCR_results` `block_level`, `collection='chronicling_america'`, revision `92477706…` (same proposed pin as S2, not yet in `config.py`); papers from `loc_chronicling_america_1770-1810` rev `0b97906a…` (= `config.py`). Needs internet.

**Rule for ranged moves (Lara):** each move counts once, in the bin of `move_before` ("arrived by"); no duplicates across bins.

**Caveats up front** (`docs/DATA.md`): move dates are ranges, often open; pension testimony is memory written 1818–1850s; newspapers skew to VA/DC/PA and post-1795; the land-ad rule is a keyword filter, not a classifier.

In [1]:
import sys, time
from pathlib import Path
PIPELINE_DIR = Path.cwd().parent
if str(PIPELINE_DIR) not in sys.path: sys.path.insert(0, str(PIPELINE_DIR))
import config
import duckdb, pandas as pd
pd.set_option('display.width',220); pd.set_option('display.max_rows',120); pd.set_option('display.max_colwidth',200)
con = duckdb.connect()
con.sql("set http_retries=12; set http_retry_wait_ms=3000; set http_retry_backoff=2")
con.sql(f"create view residences as select * from read_parquet('{config.PROCESSED_DIR/'residences.parquet'}')")
con.sql(f"create view households as select * from read_parquet('{config.PROCESSED_DIR/'households.parquet'}')")
con.sql(open(config.PIPELINE_DIR / "schema.sql").read())
def q(sql): return con.sql(sql).df()
# DuckDB 2.0: `//` on DOUBLE is float division, not floor. Dates are whole years, so cast once and use integer `//` everywhere.
con.sql("create view moves_i as select * replace (move_after::bigint as move_after, move_before::bigint as move_before) from moves")
assert q("select count(*) n from moves where move_after<>floor(move_after) or move_before<>floor(move_before)").n[0]==0, 'non-integer dates'
assert q("select (1763::bigint//10*10) v").v[0]==1760
print('duckdb', duckdb.__version__)

duckdb 2.0.0


## 1. Moves (`moves` view)
`move_after` = date of the earlier residence row, `move_before` = date of the later one. `to_kind` says where `move_before` comes from:
`removed_to` = a dated "removed to" statement (a real move date), `now_resides` = the application year (a ceiling, not a move date).

In [2]:
q('''select count(*) moves, count(distinct NAID) files,
  count(move_before) has_before, count(move_after) has_after,
  count(*) filter (where move_after is not null and move_before is not null) both_ends,
  count(*) filter (where move_after is null and move_before is not null) only_before,
  count(*) filter (where move_after is not null and move_before is null) only_after,
  count(*) filter (where move_after is null and move_before is null) undated
  from moves_i''')

,moves,files,has_before,has_after,both_ends,only_before,only_after,undated
0,18910,14235,11493,9364,5577,5916,3787,3630


In [3]:
q('''select to_kind, count(*) moves, count(move_before) has_before, count(*) filter (where move_after is not null and move_before is not null) both_ends,
  round(median(move_before-move_after)) median_width
  from moves_i group by 1 order by 2 desc''')

,to_kind,moves,has_before,both_ends,median_width
0,now_resides,11315,9856,4297,70.0
1,removed_to,5649,1145,862,29.0
2,enlisted,1142,246,224,32.0
3,married,592,170,119,23.0
4,born,122,68,68,2.0
5,lived_since,90,8,7,23.0


### 1a. Distribution of `move_after`, `move_before`, width (1763–1840 window, plus what falls outside)

In [4]:
win = q('''select
  count(*) filter (where move_before < 1763) before_1763,
  count(*) filter (where move_before between 1763 and 1840) in_1763_1840,
  count(*) filter (where move_before > 1840) after_1840,
  count(*) filter (where move_before is null) no_before
  from moves_i''')
display(win)
dec = q('''select (move_before//10*10)::int as decade, count(*) moves_by_before,
   count(*) filter (where to_kind='removed_to') removed_to, count(*) filter (where to_kind='now_resides') now_resides,
   count(*) filter (where to_kind not in ('removed_to','now_resides')) other_kind
   from moves_i where move_before between 1760 and 1849 group by 1 order by 1''')
aft = q('''select (move_after//10*10)::int as decade, count(*) moves_by_after from moves_i where move_after between 1760 and 1849 group by 1 order by 1''')
dec.merge(aft, on='decade', how='outer').sort_values('decade')

,before_1763,in_1763_1840,after_1840,no_before
0,57,9326,2110,7417


,decade,moves_by_before,removed_to,now_resides,other_kind,moves_by_after
0,1760,46,17,0,29,2492
1,1770,253,185,0,68,643
2,1780,309,194,0,115,985
3,1790,173,99,0,74,451
4,1800,205,159,0,46,323
5,1810,305,166,87,52,327
6,1820,367,129,210,28,232
7,1830,7570,156,7394,20,254
8,1840,1434,16,1401,17,69


In [5]:
# year-by-year around the spikes: how much of move_before is just the application year?
q('''select move_before::int as yr, count(*) moves, count(*) filter (where to_kind='now_resides') now_resides,
  round(100.0*count(*) filter (where to_kind='now_resides')/count(*)) pct_now_resides
  from moves_i where move_before between 1780 and 1845 group by 1 order by 1''').set_index('yr').T

yr,1780,1781,1782,1783,1784,1785,1786,1787,1788,1789,...,1836,1837,1838,1839,1840,1841,1842,1843,1844,1845
moves,64.0,53.0,35.0,23.0,27.0,16.0,19.0,32.0,23.0,17.0,...,320.0,268.0,647.0,258.0,117.0,95.0,96.0,264.0,201.0,137.0
now_resides,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,314.0,255.0,639.0,253.0,112.0,89.0,92.0,264.0,197.0,133.0
pct_now_resides,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,98.0,95.0,99.0,98.0,96.0,94.0,96.0,100.0,98.0,97.0


In [6]:
# width, for moves with both ends
w = q('''select (move_before-move_after) w from moves_i where move_after is not null and move_before is not null''').w
print('n with both ends:', len(w), '| negative widths:', int((w<0).sum()))
print(w.describe(percentiles=[.1,.25,.5,.75,.9]).round(1).to_dict())
pd.cut(w, [-1000,-1,0,2,5,10,20,40,80,1000], labels=['<0','0','1-2','3-5','6-10','11-20','21-40','41-80','>80']).value_counts().sort_index().to_frame('moves').assign(pct=lambda x:(100*x.moves/x.moves.sum()).round(1))

n with both ends: 5577 | negative widths: 98
{'count': 5577.0, 'mean': 55.3, 'std': 27.1, 'min': -80.0, '10%': 14.0, '25%': 38.0, '50%': 65.0, '75%': 74.0, '90%': 80.0, 'max': 154.0}


,moves,pct
w,,
<0,98,1.8
0,134,2.4
1-2,66,1.2
3-5,83,1.5
6-10,103,1.8
11-20,325,5.8
21-40,660,11.8
41-80,3559,63.8
>80,549,9.8


### 1b. How many moves fit inside one bin?
Containment test on `[move_after, move_before]` for moves with both ends (open ranges cannot be tested; counted separately).
Decade bins start at the decade (1770–79); 5-year bins at 1765, 1770, … (5-year: `floor(year/5)`).
Act periods (defined by when testimony was given, so they track `move_before` spikes, not movement): ≤1783 · 1784–1817 · 1818–1831 · 1832–1835 · 1836–1840 · 1841+.

In [7]:
con.sql('''create or replace macro actp(y) as case when y is null then null when y<=1783 then '1 <=1783' when y<=1817 then '2 1784-1817' when y<=1831 then '3 1818-1831'
  when y<=1835 then '4 1832-1835' when y<=1840 then '5 1836-1840' else '6 1841+' end''')
con.sql("create or replace view mv as select *, move_before-move_after w from moves_i")
cont = q('''select
  count(*) both_ends,
  count(*) filter (where move_after//10 = move_before//10) in_one_decade,
  count(*) filter (where move_after//5 = move_before//5) in_one_5yr,
  count(*) filter (where actp(move_after)=actp(move_before)) in_one_act_period,
  count(*) filter (where move_before//10 - move_after//10 = 1) spans_2_decades,
  count(*) filter (where move_before//10 - move_after//10 >= 2) spans_3plus_decades
  from mv where move_after is not null and move_before is not null''')
for c in ['in_one_decade','in_one_5yr','in_one_act_period','spans_2_decades','spans_3plus_decades']: cont[c+'_pct']=(100*cont[c]/cont.both_ends).round(1)
display(cont.T)
# same, for ranges whose end falls in 1763-1840 (the slider window)
cont2 = q('''select count(*) both_ends_1763_1840,
  round(100.0*count(*) filter (where move_after//10 = move_before//10)/count(*),1) pct_one_decade,
  round(100.0*count(*) filter (where move_after//5 = move_before//5)/count(*),1) pct_one_5yr,
  round(100.0*count(*) filter (where actp(move_after)=actp(move_before))/count(*),1) pct_one_act,
  round(median(w)) median_width
  from mv where move_after is not null and move_before between 1763 and 1840''')
cont2

,0
both_ends,5577.0
in_one_decade,266.0
in_one_5yr,214.0
in_one_act_period,620.0
spans_2_decades,252.0
spans_3plus_decades,4980.0
in_one_decade_pct,4.8
in_one_5yr_pct,3.8
in_one_act_period_pct,11.1
spans_2_decades_pct,4.5


,both_ends_1763_1840,pct_one_decade,pct_one_5yr,pct_one_act,median_width
0,4659,4.5,3.6,11.4,68.0


### 1c. Flows per bin: "arrived by `move_before`" vs range midpoint
Midpoint needs both ends, so it is computed only for the 5,577 both-ended moves; the like-for-like comparison is on that subset. The `move_before` rule can also place the 5,916 moves that have only an end date.
Counts below are for 1763–1840 only (outside = dropped, shown in 1a).

In [8]:
con.sql('''create or replace view mvb as select *, case when move_after is not null and move_before is not null then (move_after+move_before)//2 end mid from moves_i''')
both = "move_after is not null and move_before is not null"
dec_cmp = q(f'''with a as (select (move_before//10*10)::int as decade, count(*) before_rule_all_with_end,
    count(*) filter (where {both}) before_rule_both_ended from mvb where move_before between 1760 and 1849 group by 1),
  m as (select (mid//10*10)::int as decade, count(*) midpoint_rule_both_ended from mvb where mid between 1760 and 1849 group by 1)
  select * from a full join m using (decade) order by 1''')
dec_cmp

,decade,before_rule_all_with_end,before_rule_both_ended,midpoint_rule_both_ended
0,1760,46,40,247
1,1770,253,201,309
2,1780,309,238,407
3,1790,173,141,2410
4,1800,205,166,720
5,1810,305,195,729
6,1820,367,206,370
7,1830,7570,3431,229
8,1840,1434,536,75


In [9]:
# 5-year bins and Act periods under the move_before rule (all moves with an end date), and the midpoint view on the both-ended subset
b5 = q('''select (move_before//5*5)::int as bin5, count(*) moves, count(*) filter (where move_after is not null) with_start from moves_i where move_before between 1763 and 1840 group by 1 order by 1''')
b5['pct']=(100*b5.moves/b5.moves.sum()).round(1)
display(b5)
act = q('''select actp(move_before) act_period, count(*) moves, count(*) filter (where to_kind='now_resides') now_resides from moves_i where move_before is not null group by 1 order by 1''')
display(act)
mid5 = q('''select ((move_after+move_before)//2//5*5)::int as bin5_midpoint, count(*) moves from moves_i where move_after is not null and move_before is not null and (move_after+move_before)//2 between 1763 and 1840 group by 1 order by 1''')
mid5

,bin5,moves,with_start,pct
0,1760,8,7,0.1
1,1765,19,17,0.2
2,1770,42,32,0.5
3,1775,211,169,2.3
4,1780,202,155,2.2
5,1785,107,83,1.1
6,1790,85,66,0.9
7,1795,88,75,0.9
8,1800,114,99,1.2
9,1805,91,67,1.0


,act_period,moves,now_resides
0,1 <=1783,512,0
1,2 1784-1817,680,0
2,3 1818-1831,559,317
3,4 1832-1835,6022,5913
4,5 1836-1840,1610,1573
5,6 1841+,2110,2053


,bin5_midpoint,moves
0,1760,36
1,1765,176
2,1770,152
3,1775,157
4,1780,147
5,1785,260
6,1790,942
7,1795,1468
8,1800,303
9,1805,417


In [10]:
# no duplicate counting: every move with move_before is in exactly one bin
n_with = q('select count(*) n from moves_i where move_before is not null').n[0]
n_dec  = q('select count(*) n from (select move_before//10 d from moves_i where move_before is not null) ').n[0]
print('moves with move_before:', n_with, '| counted once across all decade bins:', n_dec, '| in 1763-1840:', q('select count(*) n from moves_i where move_before between 1763 and 1840').n[0])
# only moves whose end is a real move date (removed_to), i.e. not the application year
rt = q('''select (move_before//10*10)::int as decade, count(*) removed_to_moves from moves_i where to_kind='removed_to' and move_before between 1760 and 1849 group by 1 order by 1''')
print('removed_to moves with an end date in 1763-1840:', int(rt[(rt.decade>=1760)&(rt.decade<=1840)].removed_to_moves.sum()))
rt

moves with move_before: 11493 | counted once across all decade bins: 11493 | in 1763-1840: 9326
removed_to moves with an end date in 1763-1840: 1121


,decade,removed_to_moves
0,1760,17
1,1770,185
2,1780,194
3,1790,99
4,1800,159
5,1810,166
6,1820,129
7,1830,156
8,1840,16


### 1d. Households: `application_year` and first-move range
`application_year` is the declaration year; the 1818 / 1832 / 1836+ spikes are the Pension Acts, not movement.

In [11]:
ay = q('''select application_year yr, count(*) households, count(*) filter (where moved) moved from households where application_year is not null group by 1 order by 1''')
print('households', q('select count(*) n from households').n[0], '| with application_year', int(ay.households.sum()), '| range', ay.yr.min(), ay.yr.max())
top = ay.sort_values('households', ascending=False).head(10); display(top.assign(pct=lambda x:(100*x.households/ay.households.sum()).round(1)))
ay[ay.yr.between(1815,1840)].set_index('yr').T

households 78745 | with application_year 61252 | range 1818 1869


,yr,households,moved,pct
14,1832,16006,8275,26.1
2,1820,5878,2938,9.6
30,1848,3981,1969,6.5
15,1833,3610,2042,5.9
20,1838,3119,1428,5.1
25,1843,2730,1267,4.5
18,1836,2147,852,3.5
35,1853,2127,1199,3.5
0,1818,1572,616,2.6
19,1837,1567,666,2.6


yr,1818,1819,1820,1821,1822,1823,1824,1825,1826,1827,...,1831,1832,1833,1834,1835,1836,1837,1838,1839,1840
households,1572,517,5878,797,328,574,296,244,305,225,...,171,16006,3610,1562,648,2147,1567,3119,1044,588
moved,616,219,2938,417,152,304,155,137,179,109,...,93,8275,2042,888,412,852,666,1428,562,347


In [12]:
hb = q('''select case when application_year<1818 then 'a <1818' when application_year<=1831 then 'b 1818-1831' when application_year<=1835 then 'c 1832-1835'
  when application_year<=1840 then 'd 1836-1840' else 'e 1841+' end act_period, count(*) households, count(*) filter (where moved) moved
  from households where application_year is not null group by 1 order by 1''')
hb['pct']=(100*hb.households/hb.households.sum()).round(1); display(hb)
q('''select (application_year//10*10)::int as decade, count(*) households, count(*) filter (where moved) moved from households where application_year is not null group by 1 order by 1''').T

,act_period,households,moved,pct
0,b 1818-1831,12097,5931,19.7
1,c 1832-1835,21826,11617,35.6
2,d 1836-1840,8465,3855,13.8
3,e 1841+,18864,10126,30.8


,0,1,2,3,4,5
decade,1810,1820,1830,1840,1850,1860
households,2089,9594,30117,12249,6672,531
moved,835,4875,15346,6327,3838,308


In [13]:
# the household-level first move range (A08 8.5): residence chain vs the wide default 1783 -> application year
q('''select move_range_method, count(*) households,
  round(median(first_move_before-first_move_after)) median_width,
  count(*) filter (where first_move_before//10 = first_move_after//10) in_one_decade,
  count(*) filter (where first_move_before//5 = first_move_after//5) in_one_5yr
  from households where first_move_before is not null group by 1''')

,move_range_method,households,median_width,in_one_decade,in_one_5yr
0,residence_chain,10355,69.0,201,163
1,service_to_application,22294,50.0,0,0


In [14]:
q('''select move_range_method, (first_move_before//10*10)::int as decade, count(*) households from households
  where first_move_before between 1763 and 1849 group by 1,2 order by 2,1''').pivot(index='decade', columns='move_range_method', values='households').fillna(0).astype(int)

move_range_method,residence_chain,service_to_application
decade,,
1760,25,0
1770,242,0
1780,291,0
1790,160,0
1800,175,0
1810,278,823
1820,336,4757
1830,6944,9777
1840,1193,4129


## 2. Land-sale blocks: acreage, price, both
Rule = S2 / prototype 2: `(\d+ acres | tract of land | lands? for sale)` AND `(for sale | to be sold | will be sold | at private sale | public sale)` on lower-cased `block_text`; non-newspaper file `parse_backup_0` dropped.
- **has acres**: `\d[\d,.]* acre(s)` (digits directly before the word; OCR variants like "acrcs" and spelled numbers are missed).
- **has price**: a currency mark attached to a number: `£ 5`, `$5`, or `<number> pounds|shillings|dollars|pence|cents` (`pounds of / weight` removed first: tobacco, hay).
- **explicit $/acre**: a price **and** "per acre" / "an acre" / "the acre" / "/acre" in the same block. Totals are not divided by acres here (no pairing of which number is which).
- Currency words are also counted on their own (not number-attached), since "dollars" can be a reward or "pounds" a weight.

In [15]:
HF='hf://datasets/RevolutionCrossroads/'
NEWS_GLOB=f"{HF}loc_chronicling_america_1770-1810@{config.DATASETS['news']['revision']}/data/train-*.parquet"
OCR_REV='92477706d19db4dcd0c425e01e1ff85a209a8269'
BLOCK_GLOB=f"{HF}RevX_OCR_results@{OCR_REV}/block_level/train-*.parquet"
t=time.time()
con.sql(f'''create or replace table papers as select lccn, any_value(newspaper_title) title, any_value(place_of_publication) place, min(year(issue_date)) first_yr, max(year(issue_date)) last_yr
  from read_parquet('{NEWS_GLOB}') group by 1''')
print(len(q('select * from papers')), 'papers', round(time.time()-t,1), 's')
con.sql("""create or replace macro paperst(p) as case when p ilike '%Va.%' or p ilike '%Virginia%' then 'VA' when p ilike '%Pa.%' then 'PA' when p ilike '%N.H.%' then 'NH'
  when p ilike '%Md.%' then 'MD' when p ilike '%Del%' then 'DE' when p ilike '%N.J.%' then 'NJ' when p ilike '%N.Y.%' then 'NY' when p ilike '%Ky.%' then 'KY'
  when p ilike '%Me.%' then 'ME' when p ilike '%D.C.%' then 'DC' when p ilike '%Mass%' then 'MA' when p ilike '%Conn%' then 'CT' when p ilike '%Vt%' then 'VT'
  when p ilike '%Ohio%' then 'OH' when p ilike '%Tenn%' then 'TN' when p ilike '%S.C.%' then 'SC' when p ilike '%N.C.%' then 'NC' when p ilike '%Ga.%' then 'GA' when p ilike '%La.%' then 'LA' else null end""")
print('papers with no state from the macro:', q('select count(*) n from papers where paperst(place) is null').n[0])

110 papers 1.7 s
papers with no state from the macro: 6


In [16]:
RE_A=r"\d+ acres|tract of land|lands? for sale"
RE_B=r"for sale|to be sold|will be sold|at private sale|public sale"
RE_ACRES=r"\d[\d,.]*\s*acres?"
RE_MONEY=r"(£|\$)\s?\d|\d[\d,.]*\s*(pounds|shillings|dollars|pence|cents|dols)"
RE_UNIT=r"per acre|an acre|the acre|/\s*acre|per cent\.? per acre"
t=time.time()
con.sql(f'''create or replace table land as
 with s as (select filename, page_number, block_id, block_type, block_text, lower(block_text) lt,
     regexp_replace(lower(block_text), 'pounds\\s+(of|weight)', '', 'g') lt2
   from read_parquet('{BLOCK_GLOB}') where collection='chronicling_america' and filename like 'sn%'
   and regexp_matches(lower(block_text),'{RE_A}') and regexp_matches(lower(block_text),'{RE_B}'))
 select filename, page_number, block_id, block_type, block_text, length(block_text) len,
   split_part(filename,'_',1) lccn, split_part(filename,'_',2) d, year(try_cast(split_part(filename,'_',2) as date)) yr,
   regexp_matches(lt,'{RE_ACRES}') has_acres,
   regexp_matches(lt2,'{RE_MONEY}') has_price,
   regexp_matches(lt2,'{RE_UNIT}') has_unit,
   contains(lt,'£') w_pound_sign, regexp_matches(lt2,'\\bpounds\\b') w_pounds, regexp_matches(lt,'\\bshillings?\\b') w_shillings,
   regexp_matches(lt,'\\bdollars?\\b|\\bdols?\\b') w_dollars, contains(lt,'$') w_dollar_sign,
   regexp_matches(lt,'\\bcurrency\\b|\\bspecie\\b|\\bsterling\\b') w_currency_type
 from s''')
print('scan', round(time.time()-t,1), 's')
con.sql("create or replace table land2 as select l.*, p.place, paperst(p.place) st, p.title from land l left join papers p using (lccn)")
q('select count(*) land_blocks, count(distinct (lccn,d,page_number)) pages, min(yr) min_yr, max(yr) max_yr, count(*) filter (where place is null) no_paper from land2')

scan 22.4 s


,land_blocks,pages,min_yr,max_yr,no_paper
0,26117,15071,1770,1810,0


### 2a. Headline: acres, price, both

In [17]:
cov = q('''select count(*) land_blocks,
  count(*) filter (where has_acres) with_acres,
  count(*) filter (where has_price) with_price,
  count(*) filter (where has_acres and has_price) with_both,
  count(*) filter (where has_acres and not has_price) acres_only,
  count(*) filter (where has_price and not has_acres) price_only,
  count(*) filter (where not has_acres and not has_price) neither,
  count(*) filter (where has_acres and has_price and has_unit) both_and_per_acre,
  count(*) filter (where has_unit) per_acre_any
  from land2''')
c0 = cov.iloc[0]
display(cov.T.assign(pct_of_blocks=lambda x:(100*x[0]/c0.land_blocks).round(1)))

,0,pct_of_blocks
land_blocks,26117,100.0
with_acres,18686,71.5
with_price,975,3.7
with_both,803,3.1
acres_only,17883,68.5
price_only,172,0.7
neither,7259,27.8
both_and_per_acre,101,0.4
per_acre_any,526,2.0


### 2a-bis. Repeats: blocks are not distinct ads
Ads usually run in more than one issue (DATA.md), so a block count overstates ads. Distinct-text estimate: key = first 150 letters/digits of the lower-cased text. It still splits reprints whose date line changed and merges different ads on one template, so read it as a rough range.

In [18]:
ded = q('''with k as (select *, regexp_replace(lower(substr(block_text,1,300)),'[^a-z0-9]','','g')[1:150] key from land2)
 select count(distinct key) distinct_texts,
  count(distinct key) filter (where has_acres) with_acres,
  count(distinct key) filter (where has_price) with_price,
  count(distinct key) filter (where has_acres and has_price) with_both,
  count(distinct key) filter (where has_acres and has_price and has_unit) both_and_per_acre from k''')
ded.T.rename(columns={0:'distinct_texts'})

,distinct_texts
distinct_texts,11792
with_acres,9420
with_price,626
with_both,516
both_and_per_acre,57


### 2b. By year, and by decade / 5-year bin / Act period (blocks count once, by issue year)

In [19]:
by_yr = q('''select yr, count(*) land_blocks, count(*) filter (where has_acres) acres, count(*) filter (where has_price) price,
  count(*) filter (where has_acres and has_price) both_, count(*) filter (where has_acres and has_price and has_unit) both_per_acre from land2 group by 1 order by 1''')
by_yr['pct_both']=(100*by_yr.both_/by_yr.land_blocks).round(1)
by_yr

,yr,land_blocks,acres,price,both_,both_per_acre,pct_both
0,1770,125,120,0,0,0,0.0
1,1771,159,53,0,0,0,0.0
2,1772,215,63,0,0,0,0.0
3,1773,499,401,2,2,0,0.4
4,1774,610,501,5,5,0,0.8
5,1775,336,314,5,5,0,1.5
6,1776,501,476,2,2,0,0.4
7,1777,392,344,1,1,0,0.3
8,1778,156,130,0,0,0,0.0
9,1779,369,304,0,0,0,0.0


In [20]:
con.sql("create or replace view land3 as select *, (yr//10*10) dec, (yr//5*5) b5 from land2")
agg = '''count(*) land_blocks, count(*) filter (where has_acres) acres, count(*) filter (where has_price) price,
  count(*) filter (where has_acres and has_price) both_, count(*) filter (where has_acres and has_price and has_unit) both_per_acre, count(distinct lccn) papers'''
d1=q(f'select dec bin, {agg} from land3 group by 1 order by 1'); d2=q(f'select b5 bin, {agg} from land3 group by 1 order by 1')
d3=q(f"select case when yr<=1783 then '<=1783' when yr<=1817 then '1784-1817' when yr<=1831 then '1818-1831' else '1832+' end bin, {agg} from land3 group by 1 order by 1")
for n,d in [('decade',d1),('5-year',d2),('Act-period (news years; no ads after 1810)',d3)]:
    print(n); display(d.assign(pct_both=(100*d.both_/d.land_blocks).round(1)))

decade


,bin,land_blocks,acres,price,both_,both_per_acre,papers,pct_both
0,1770,3362,2706,15,15,0,25,0.4
1,1780,2567,2103,28,24,0,25,0.9
2,1790,3511,2798,71,70,2,22,2.0
3,1800,14557,9665,771,616,98,42,4.2
4,1810,2120,1414,90,78,1,11,3.7


5-year


,bin,land_blocks,acres,price,both_,both_per_acre,papers,pct_both
0,1770,1608,1138,7,7,0,9,0.4
1,1775,1754,1568,8,8,0,21,0.5
2,1780,1435,1286,10,6,0,14,0.4
3,1785,1132,817,18,18,0,13,1.6
4,1790,623,456,2,2,0,13,0.3
5,1795,2888,2342,69,68,2,17,2.4
6,1800,4225,2866,169,146,18,32,3.5
7,1805,10332,6799,602,470,80,21,4.5
8,1810,2120,1414,90,78,1,11,3.7


Act-period (news years; no ads after 1810)


,bin,land_blocks,acres,price,both_,both_per_acre,papers,pct_both
0,1784-1817,21509,14850,950,782,101,73,3.6
1,<=1783,4608,3836,25,21,0,30,0.5


### 2c. Currency words by year and by state

In [21]:
cw = ['w_pound_sign','w_pounds','w_shillings','w_dollars','w_dollar_sign','w_currency_type']
sel = ', '.join(f"count(*) filter (where {c}) {c[2:]}" for c in cw)
cy = q(f"select yr, count(*) land_blocks, {sel}, count(*) filter (where not ({' or '.join(cw)})) no_currency_word from land2 group by 1 order by 1")
cy

,yr,land_blocks,pound_sign,pounds,shillings,dollars,dollar_sign,currency_type,no_currency_word
0,1770,125,0,5,0,0,0,0,120
1,1771,159,0,11,6,0,0,0,143
2,1772,215,0,7,2,3,0,0,203
3,1773,499,0,11,5,0,0,0,483
4,1774,610,2,4,5,1,0,8,594
5,1775,336,16,13,4,3,1,4,302
6,1776,501,0,17,7,1,0,9,471
7,1777,392,1,4,4,0,0,14,373
8,1778,156,0,0,0,2,0,0,154
9,1779,369,0,8,0,3,1,12,345


In [22]:
cd = q(f"select dec, count(*) land_blocks, {sel}, count(*) filter (where not ({' or '.join(cw)})) no_currency_word from land3 group by 1 order by 1")
cd

,dec,land_blocks,pound_sign,pounds,shillings,dollars,dollar_sign,currency_type,no_currency_word
0,1770,3362,19,80,33,13,2,47,3188
1,1780,2567,8,219,136,21,17,53,2258
2,1790,3511,14,32,34,55,62,68,3284
3,1800,14557,215,254,108,1065,147,57,13034
4,1810,2120,16,19,8,131,23,5,1949


In [23]:
cs = q(f"select coalesce(st,'?') state, count(*) land_blocks, {sel}, count(*) filter (where has_acres and has_price) both_ from land2 group by 1 order by 2 desc")
cs['pct_both']=(100*cs.both_/cs.land_blocks).round(1); cs

,state,land_blocks,pound_sign,pounds,shillings,dollars,dollar_sign,currency_type,both_,pct_both
0,VA,13353,180,231,70,808,137,59,474,3.5
1,PA,3774,14,56,47,58,58,100,70,1.9
2,DC,2668,1,53,0,211,20,0,118,4.4
3,KY,1725,47,31,66,85,8,14,70,4.1
4,NY,1256,18,183,93,15,17,33,21,1.7
5,NH,1167,4,18,10,48,1,11,10,0.9
6,NJ,568,4,28,28,4,0,0,0,0.0
7,ME,557,1,0,0,49,0,0,25,4.5
8,?,500,1,0,0,7,10,13,12,2.4
9,CT,269,1,3,5,0,0,0,2,0.7


In [24]:
# decade x state (top states), blocks with acres AND price
top_st = list(cs.state.head(6))
q(f"select dec, coalesce(st,'?') state, count(*) filter (where has_acres and has_price) both_ from land3 where coalesce(st,'?') in ({','.join(repr(s) for s in top_st)}) group by 1,2").pivot(index='dec', columns='state', values='both_').fillna(0).astype(int)

state,DC,KY,NH,NY,PA,VA
dec,,,,,,
1770,0,0,2,4,2,5
1780,0,1,0,17,5,0
1790,0,5,3,0,61,1
1800,118,64,5,0,2,394
1810,0,0,0,0,0,74


### 2d. Ten samples with acres and a price (ordered by hash: reproducible)

In [25]:
s = q('''select l.lccn, l.place, l.d, l.page_number+1 page, l.has_unit, replace(replace(substr(l.block_text,1,420),chr(10),' '),chr(13),' ') snippet
  from land2 l where has_acres and has_price order by hash(l.filename||l.block_id) limit 10''')
for i,r in s.iterrows(): print(f"[{i}] {r.lccn} {r.place} {r.d} p{r.page} per_acre_wording={r.has_unit}\n    {r.snippet}\n")

[0] sn83045242 Washington City [D.C.] 1802-04-26 p3 per_acre_wording=False
    A TRACT or parcel of land consisting of 240 acres, more or less, late the property of Robert Buchanan, deceased; lying in Prince George County, Maryland, and on the point branch of the Eastern Branch of the Potomac River. Any person inclined to purchase, may view the wood and meadow land, as also the improvements and situation. If the above land is not sold before the 25th day of August, it then will be offered for p

[1] sn84024710 Richmond, Va. 1808-11-08 p4 per_acre_wording=False
    FOR SALE, THE subscriber offers for SALE, the TRACT OF LAND, on which he now resides, lying on Ward's fork, about two miles from its junction with Rappahannock, from Charlottesville, and containing 300 acres, about 120 of which is cleared and inclosed with good fencing. The improvements are a good dwelling house, with 2 brick chimneys, a grist mill with 2 pair of stones, and a saw mill, on a stream which has never be

[2] sn8

### 2e. Known false-positive patterns

In [26]:
q('''select count(*) both_blocks,
 count(*) filter (where len>3000) over_3000_chars,
 count(*) filter (where regexp_matches(lower(block_text),'\\b(to let|to rent|for rent|lease)\\b')) rent_lease_wording,
 count(*) filter (where regexp_matches(lower(block_text),'reward|ran away|stolen|strayed|run away')) reward_runaway_wording,
 count(*) filter (where regexp_matches(lower(block_text),'tobacco|hhds|hogsheads|flour|bushels')) goods_wording,
 count(*) filter (where regexp_matches(lower(block_text),'sheriff|execution|trustee|administrator|commissioner|decree')) legal_sale_wording,
 count(*) filter (where has_unit) per_acre_wording
 from land2 where has_acres and has_price''')

,both_blocks,over_3000_chars,rent_lease_wording,reward_runaway_wording,goods_wording,legal_sale_wording,per_acre_wording
0,803,94,86,29,174,206,101


In [27]:
# blocks with explicit per-acre wording: read the first 8 (are they land prices?)
s = q('''select l.d, l.place, replace(substr(block_text, greatest(1, coalesce(nullif(strpos(lower(block_text),'per acre'),0), nullif(strpos(lower(block_text),'an acre'),0), 1)-120), 300),chr(10),' ') around
  from land2 l where has_acres and has_price and has_unit order by hash(filename||block_id) limit 8''')
for i,r in s.iterrows(): print(f"[{i}] {r.d} {r.place}: ...{r.around}...\n")

[0] 1809-11-07 Richmond, Va.: ... eligible stand for any person wishing to turn his attention to public business. The price required is only six dollars per acre, which is considerably under its value, in cash—but if time is wished it can be given as far as three or four years upon payment of legal interest. The sale will be made p...

[1] 1798-05-14 Philadelphia [Pa.]: ...ile Run. These Lands are fertile and well timbered, and abound in Coal; they were sold ten years ago for five shillings per acre; patented early in 1787, except 3700 acres, which were patented in 1798; part of them were surveyed in 1798. The tract is between the Ohio and Monongahela, very convenient...

[2] 1809-10-10 Richmond, Va.: ... eligible stanc for any person wishing to turn his attention to public business. The price required is only six dollars per acre, which is considerably under its value, in cash— but if time is wished it can be given as far as three or four years upon payment of le- gal interest. The sa

## 3. Assertions

In [28]:
assert q('select count(*) n from moves_i').n[0]==18910, 'moves count differs from DATA.md'
assert q('select count(*) n from land2').n[0]>0
assert q('select count(*) n from land2').n[0]==q('select count(*) n from land').n[0], 'paper join duplicated rows'
print('ok')

ok


## 4. Recommendation
Numbers are from the tables above (1763–1840 window for moves; 1770–1810 for ads, the corpus range).

1. **Decades as the slider spine** (1770s … 1830s for moves; 1770s–1800s plus a one-year 1810 for ads). Ads: 3,362 / 2,567 / 3,511 / 14,557 / 2,120 land-sale blocks per decade, every cell large enough; 5-year bins drop to 623 in 1790–94 and the 1810 bin is one year only. Moves: 5-year bins hold 8–250 moves each before 1830 (see 1c), too thin to read.
2. **No bin can say a move "happened in" it**: for the 5,577 both-ended moves the median range is 65 years; only 4.8% fit in one decade, 3.8% in one 5-year bin (11% in one Act period). Label the layer "arrived by", as Lara's rule says, and do not use the range midpoint: it piles 2,410 of 5,577 into the 1790s from born → now-resides ranges that say nothing about 1790s movement.
3. **The 1830s bin is application years, not movement**: 7,570 of the 9,326 moves ending 1763–1840 (81%) fall in 1830–39, and 7,394 of those end at the application year (`now_resides`). Only 1,121 moves in the window end at a dated "removed to". Show the household layer as **application cohort by Act period** (1818–31: 12,097 · 1832–35: 21,826 · 1836–40: 8,465 · 1841+: 18,864 households; 1832 alone is 16,006 = 26%, 1820 is 5,878) and mark the Act years on the slider; use the 1,121 dated moves for any "when" claim.
4. **$/acre is a footnote, not a layer**: of 26,117 land-sale blocks, 18,686 (71.5%) name acres, 975 (3.7%) a price, 803 (3.1%) both, and only 101 (0.4%) say a price per acre outright; by distinct text (repeats collapsed) that is 11,792 ads, 9,420 with acres, 626 with a price, 516 with both, 57 with a per-acre price. 616 of the 803 are 1800–09 and 474 are Virginia papers; no price at all before 1773, almost none before 1796.
5. **Currency**: dollars dominate from 1802 (1,065 blocks with "dollars" in the 1800s vs 254 "pounds", 108 "shillings"); £/pounds/shillings cluster in 1784–88 and in KY/NY blocks. Before 1796 the sample is too small for any price series. If a price view is wanted, pair price with acreage by hand-checking ~800 blocks, or drop it.